NOTE: The "Check" function is used to check is any problems are found within your dataframe

In [1]:
import pandas as pd
import numpy as np
from plotnine import*
import calendar

In [2]:
df_airquality = pd.read_csv('Air_quality_analytics.csv')

In [3]:
# Converting date column as a date format

df_airquality['date'] = pd.to_datetime(df_airquality['date'], format = "%Y-%m-%d")
df_airquality.info()
df_airquality.head()

<class 'pandas.DataFrame'>
RangeIndex: 87084 entries, 0 to 87083
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   _id              87084 non-null  int64         
 1   date             87084 non-null  datetime64[us]
 2   site             87084 non-null  str           
 3   parameter        87084 non-null  str           
 4   index_value      87084 non-null  int64         
 5   description      87084 non-null  str           
 6   health_advisory  7573 non-null   str           
 7   health_effects   7573 non-null   str           
dtypes: datetime64[us](1), int64(2), str(5)
memory usage: 5.3 MB


,_id,date,site,parameter,index_value,description,health_advisory,health_effects
0,1,2016-01-01,Lawrenceville,PM25B,25,Good,NaN,NaN
1,2,2016-01-01,Flag Plaza,CO,0,Good,NaN,NaN
2,3,2016-01-01,Harrison Township,OZONE,31,Good,NaN,NaN
3,4,2016-01-01,Avalon,SO2,10,Good,NaN,NaN
4,5,2016-01-01,Lincoln,PM25,35,Good,NaN,NaN


In [4]:
# Getting rid of additional duplicates

df_airquality = df_airquality[df_airquality['parameter'] != 'PM25(2)']
df_airquality = df_airquality[df_airquality['site'] != 'Lawrenceville 2']
df_airquality = df_airquality[df_airquality['site'] != 'Liberty 2']

In [5]:
# Deleting partial year 2026

df_airquality = df_airquality[df_airquality['date'].dt.year != 2026]

In [6]:
# Checking for Duplicates

n_rows = len(df_airquality)
n_unique = df_airquality[['site', 'parameter', 'date']].drop_duplicates().shape[0]

print(n_rows, n_unique)
print(n_rows == n_unique)

# No Duplicates Confirmed

66304 66304
True


In [7]:
# Adding month & month name column

df_airquality['month'] = df_airquality['date'].dt.month
df_airquality['month_name'] = df_airquality['month'].apply(lambda x: calendar.month_name[x])

In [8]:
# Adding Season Column

season = {1: 'Winter', 2: 'Winter', 3: 'Spring', 4: 'Spring', 5: 'Spring', 6: 'Summer', 7: 'Summer', 8: 'Summer', 9: 'Fall', 10: 'Fall', 11: 'Fall', 12: 'Winter'}
df_airquality['season'] = df_airquality['month'].map(season)

In [9]:
df_airquality.info()
df_airquality.head()

<class 'pandas.DataFrame'>
Index: 66304 entries, 0 to 80730
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   _id              66304 non-null  int64         
 1   date             66304 non-null  datetime64[us]
 2   site             66304 non-null  str           
 3   parameter        66304 non-null  str           
 4   index_value      66304 non-null  int64         
 5   description      66304 non-null  str           
 6   health_advisory  5313 non-null   str           
 7   health_effects   5313 non-null   str           
 8   month            66304 non-null  int32         
 9   month_name       66304 non-null  str           
 10  season           66304 non-null  str           
dtypes: datetime64[us](1), int32(1), int64(2), str(7)
memory usage: 5.8 MB


,_id,date,site,parameter,index_value,description,health_advisory,health_effects,month,month_name,season
0,1,2016-01-01,Lawrenceville,PM25B,25,Good,NaN,NaN,1,January,Winter
1,2,2016-01-01,Flag Plaza,CO,0,Good,NaN,NaN,1,January,Winter
2,3,2016-01-01,Harrison Township,OZONE,31,Good,NaN,NaN,1,January,Winter
3,4,2016-01-01,Avalon,SO2,10,Good,NaN,NaN,1,January,Winter
4,5,2016-01-01,Lincoln,PM25,35,Good,NaN,NaN,1,January,Winter


In [10]:
# Reference

df_airquality['parameter'].unique()

<StringArray>
[   'PM25B',       'CO',    'OZONE',      'SO2',     'PM25',     'PM10',
    'PM10B',      'NO2',    'PM25T', 'PM25_640', 'PM10_640',  'NO2_500',
  'NO2_200']
Length: 13, dtype: str

Now we must start to group pollutants together, to make it easier for identifying domains of pollutants

#TO DO:

# Group by Pollutant and Season and take the Mean

In [11]:
# Grouping by Pollutant

pollutant_dictionary = {
    'PM25':  ['PM25', 'PM25B', 'PM25T', 'PM25_640'],
    'PM10':  ['PM10', 'PM10B', 'PM10_640'],
    'CO':    ['CO'],
    'Ozone': ['OZONE'],
    'SO2':   ['SO2'],
    'NO2':   ['NO2', 'NO2_500', 'NO2_200'],
}

In [12]:
code_to_group = {code: group
                 for group, codes in pollutant_dictionary.items()
                 for code in codes}

In [13]:
df_airquality['pollutant_dic'] = df_airquality['parameter'].map(code_to_group)

In [14]:
daily = (df_airquality.groupby(['site', 'date', 'pollutant_dic'])['index_value']
           .mean().reset_index())

<class 'pandas.DataFrame'>
Index: 66304 entries, 0 to 80730
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   _id              66304 non-null  int64         
 1   date             66304 non-null  datetime64[us]
 2   site             66304 non-null  str           
 3   parameter        66304 non-null  str           
 4   index_value      66304 non-null  int64         
 5   description      66304 non-null  str           
 6   health_advisory  5313 non-null   str           
 7   health_effects   5313 non-null   str           
 8   month            66304 non-null  int32         
 9   month_name       66304 non-null  str           
 10  season           66304 non-null  str           
 11  pollutant_dic    66304 non-null  str           
dtypes: datetime64[us](1), int32(1), int64(2), str(8)
memory usage: 6.3 MB
